# 2 · How deep should the tree be? Overfitting, knobs and thresholds

**What you practise (from the decision-tree lecture):**
- The honest test: learn on some orders, test on orders the tree has never seen.
- **Red flag #3:** a perfect training score.
- **Activity B again:** how deep? Train vs validation results for depth 1 … unlimited.
- The three **knobs**: `max_depth`, `min_samples_leaf`, `class_weight`.
- **ROC-AUC**: can the model rank late orders above on-time orders?
- **Threshold = policy**: how many orders can dispatch actually help?

> **How this notebook works**
> - Run the cells **from top to bottom** with `Shift + Enter`.
> - ▶️ **Run** cells work as they are.
> - ✏️ **Your turn** cells contain `___` (three underscores). Replace each `___` with your answer, then run the cell.
> - ✅ **Check** cells tell you whether you got it right. If a check fails, read the hint, fix the cell above and run both again.
> - 🧠 **Think** cells ask a PM question. Double-click the answer cell below it and write 1–3 sentences.
> - If you see `NameError: name '___' is not defined`, a blank is still empty.
> - Stuck? The `solutions/` folder has the same notebook, solved.

## Step 0 · Load and prepare (same as notebook 1)

In [ ]:
# ▶️ Run: load the tools and the data, and prepare the inputs exactly like notebook 1
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import roc_auc_score, ConfusionMatrixDisplay

orders = pd.read_csv("data/quickcart_orders.csv")
features = ["order_hour", "weekday", "distance_km", "items_count", "basket_value_eur", "weather",
            "traffic_level", "store_queue", "courier_experience_months", "vehicle",
            "is_premium_customer", "promo_used", "customer_previous_orders"]
X = pd.get_dummies(orders[features], dtype=int)
y = orders["late"]
print("X:", X.shape, "| late share:", round(y.mean(), 3))

## Step 1 · Keep an exam aside: the train/test split

A tree that learned from an order has *seen the answer*. To know how it does on **new** orders, we hide some orders and only use them at the very end.

In [ ]:
# ✏️ Your turn: keep 25% of the orders aside as the test set (write it as a decimal)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=___, random_state=42, stratify=y)
print("train:", len(X_train), "orders | test:", len(X_test), "orders")

In [ ]:
# ✅ Check
assert len(X_test) == 750, "test_size should be 0.25"
print("✅ Correct. 2,250 orders to learn from, 750 kept aside. Late share in test:", round(y_test.mean(), 3))
print("   stratify=y keeps the same share of late orders in both parts.")

## Step 2 · ROC-AUC in one line

**ROC-AUC** = if we pick one late order and one on-time order at random, how often does the model give the **late** one the **higher** score?
- 0.5 = coin flip · 1.0 = perfect ranking. (Ties, which trees produce a lot, count as half.)
- It measures *ranking*, not *who to call*: the threshold decides that.

## Step 3 · Red flag #3: let the tree grow without limits

In [ ]:
# ▶️ Run: a tree with NO depth limit
big_tree = DecisionTreeClassifier(random_state=42)
big_tree.fit(X_train, y_train)
train_auc = roc_auc_score(y_train, big_tree.predict_proba(X_train)[:, 1])
print("Leaves (= rules):", big_tree.get_n_leaves(), "| deepest route:", big_tree.get_depth(), "questions")
print("Train AUC:", round(train_auc, 3))

In [ ]:
# ✏️ Your turn: compute the AUC on the TEST orders (copy the train line and change two things)
test_auc = roc_auc_score(___, big_tree.predict_proba(___)[:, 1])
print("Test AUC:", round(test_auc, 3))

In [ ]:
# ✅ Check
assert abs(test_auc - 0.624) < 0.005, "Use y_test and X_test."
print("✅ Train 1.00 vs test 0.62: the tree memorised 2,250 orders with", big_tree.get_n_leaves(), "rules.")

In [ ]:
# ▶️ Run: draw the unlimited tree. Don't try to read it: that's the point!
plt.figure(figsize=(18, 7))
plot_tree(big_tree, filled=True, impurity=False, label="none", fontsize=1)
plt.title(f"Unlimited tree: {big_tree.get_n_leaves()} leaves, {big_tree.get_depth()} questions deep. Could dispatch explain THIS?")
plt.show()

🧠 **Think:** Jonas proudly reports "training AUC 1.00". What do you tell him, and what do you ask for?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 4 · How deep? (Activity B)

We try several depths. To **choose**, we must not peek at the test set: that's for the final exam only.
Instead we use **cross-validation**: the training orders are split into 5 parts, and the tree is trained 5 times, each time tested on the part it didn't see. Think of it as **5 rehearsal exams**. The average is the *validation AUC*.

In [ ]:
# ▶️ Run: try 8 depths (None = no limit). This takes a few seconds.
depths = [1, 2, 3, 4, 5, 6, 8, None]
rows = []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    validation_auc = cross_val_score(tree, X_train, y_train, cv=5, scoring="roc_auc").mean()
    tree.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, tree.predict_proba(X_train)[:, 1])
    rows.append({"max_depth": str(depth), "leaves": tree.get_n_leaves(),
                 "train_AUC": round(train_auc, 3), "validation_AUC": round(validation_auc, 3)})
sweep = pd.DataFrame(rows)
sweep

In [ ]:
# ▶️ Run: the picture of overfitting
plt.figure(figsize=(8, 4))
plt.plot(sweep["max_depth"], sweep["train_AUC"], marker="o", label="train (seen orders)")
plt.plot(sweep["max_depth"], sweep["validation_AUC"], marker="o", label="validation (unseen orders)")
plt.xlabel("max_depth"); plt.ylabel("AUC"); plt.ylim(0.5, 1.02); plt.legend(); plt.grid(alpha=0.3)
plt.title("Deeper = better on seen orders, worse on new ones")
plt.show()

In [ ]:
# ✏️ Your turn: which depth has the HIGHEST validation AUC? (a number, not text)
best_depth = ___

In [ ]:
# ✅ Check
assert best_depth in (3, 4), "Look at the validation_AUC column, not train_AUC."
print("✅ Depth 4 is best (0.773), depth 3 almost the same (0.772) with only 8 rules.")
print("   After depth 4 validation AUC falls while train AUC keeps rising: the tree starts learning noise.")

🧠 **Think:** depth 3 (8 rules, validation 0.772) or depth 4 (15 rules, 0.773)? Which would you pick for dispatch, and why?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 5 · The other two knobs

| knob | in plain language | fixes |
|---|---|---|
| `max_depth` | no order passes through more than N questions | rules too long to read |
| `min_samples_leaf` | every final group must contain at least N training orders | rules backed by 1–2 orders |
| `class_weight="balanced"` | while learning, give extra influence to the rarer class (late) | late orders getting too little attention |

In [ ]:
# ✏️ Your turn: no depth limit, but every leaf needs at least 50 training orders
knob_tree = DecisionTreeClassifier(min_samples_leaf=___, random_state=42)
knob_auc = cross_val_score(knob_tree, X_train, y_train, cv=5, scoring="roc_auc").mean()
knob_tree.fit(X_train, y_train)
print("Leaves:", knob_tree.get_n_leaves(), "| validation AUC:", round(knob_auc, 3))

In [ ]:
# ✅ Check
assert knob_tree.get_n_leaves() == 33, "Use min_samples_leaf=50"
print("✅ From 392 rules to 33, and validation AUC from 0.594 to", round(knob_auc, 3), "with ONE setting.")

In [ ]:
# ✏️ Your turn: compare recall (share of late orders flagged with the default 0.5 rule) without and
# with class_weight. Fill in the text value. We measure on the rehearsal exams (cross-validation),
# not on orders the tree learned from.
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import recall_score
for weight in [None, "___"]:
    t = DecisionTreeClassifier(max_depth=4, class_weight=weight, random_state=42)
    flagged = cross_val_predict(t, X_train, y_train, cv=5)
    print(f"class_weight={weight}: flags {flagged.sum()} of 2,250 orders, recall {recall_score(y_train, flagged):.2f}")

🧠 **Think:** class_weight raised recall a lot. Is it the same thing as lowering the threshold? What is different?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 6 · Threshold = policy. How many orders can dispatch help?

Lina: *"Dispatch can send extra help to **at most 200 out of every 750 orders**."*

We choose the threshold on the **rehearsal exams** as well: every training order gets a score from a tree that did *not* learn from it. The test set stays locked until the very end.

In [ ]:
# ▶️ Run: what each threshold would mean for dispatch (rehearsal exams on the 2,250 training orders)
rehearsal_scores = cross_val_predict(DecisionTreeClassifier(max_depth=4, random_state=42),
                                     X_train, y_train, cv=5, method="predict_proba")[:, 1]
rows = []
for th in [0.5, 0.4, 0.3, 0.2]:
    flag = rehearsal_scores >= th
    caught = (flag & (y_train == 1)).sum()
    rows.append({"threshold": th, "flags per 750 orders": round(flag.sum() * 750 / len(y_train)),
                 "recall": round(caught / y_train.sum(), 2), "precision": round(caught / flag.sum(), 2)})
pd.DataFrame(rows)

In [ ]:
# ✏️ Your turn: from the table, pick the LOWEST threshold that keeps flags per 750 orders at 200 or fewer
chosen_threshold = ___

In [ ]:
# ✅ Check
per_750 = (rehearsal_scores >= chosen_threshold).sum() * 750 / len(y_train)
assert per_750 <= 200, f"About {per_750:.0f} flags per 750 orders: more than dispatch can handle. Try a higher threshold."
if chosen_threshold > 0.4:
    print(f"Within capacity ({per_750:.0f} per 750), but 0.4 also fits and catches more late orders.")
else:
    print(f"✅ About {per_750:.0f} flags per 750 orders: within capacity. 0.3 would need about 274, too many.")
print("Note: a depth-4 tree has only ~15 different scores, so the number of flags jumps instead of changing smoothly.")

## Step 7 · The final exam: test the chosen tree and policy ONCE

In [ ]:
# ✏️ Your turn: our final model is the depth-4 tree. Train it and score the TEST orders.
final_tree = DecisionTreeClassifier(max_depth=___, random_state=42).fit(X_train, y_train)
test_scores = final_tree.predict_proba(X_test)[:, 1]
final_auc = roc_auc_score(y_test, test_scores)
print("Final test AUC:", round(final_auc, 3))

In [ ]:
# ✅ Check
assert abs(final_auc - 0.770) < 0.002, "Use max_depth=4"
print("✅ Test AUC 0.77, close to the validation estimate (0.773): the rehearsal exams were honest.")

In [ ]:
# ▶️ Run: your policy on the test orders: the confusion matrix
flag = test_scores >= chosen_threshold
caught = (flag & (y_test == 1)).sum()
ConfusionMatrixDisplay.from_predictions(y_test, flag.astype(int), display_labels=["on time", "late"], cmap="Greens")
plt.title(f"Policy: flag if score ≥ {chosen_threshold}")
plt.show()
print(f"{flag.sum()} flags · {caught} of {y_test.sum()} late orders caught · {y_test.sum() - caught} missed · {flag.sum() - caught} false alarms")

**What are we shipping?** Here is the final depth-4 tree. Every route asks at most 4 questions; colour = orange for mostly on time, blue for mostly late.

In [ ]:
# ▶️ Run: draw the final tree (zoom in if the text is small)
plt.figure(figsize=(34, 11))
plot_tree(final_tree, feature_names=list(X_train.columns), class_names=["on time", "late"],
          filled=True, rounded=True, impurity=False, fontsize=9)
plt.title(f"Final tree: depth 4, {final_tree.get_n_leaves()} leaves")
plt.show()

🧠 **Think:** with 0.4, dispatch reaches fewer than half of the late orders. Threshold 0.3 would catch far more (recall about 0.68 instead of 0.41 in the rehearsal exams) but needs about 274 flags per 750 orders. What would you propose to Lina?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## ✅ Check your understanding

1. Why do we choose the depth and the threshold with cross-validation and not with the test set?
2. What does `min_samples_leaf=50` protect you from?
3. Test AUC is 0.77. Does that tell you whom to flag?

**Your answers to 1–3:**

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

### 🎉 Great work! Next: notebook 3, *find the product failure and make the PM call.*